## Lunar Events Calculator

An astronomically accurate tool for calculating key lunar events, designed for both modern and historical research. This calculator determines:

- Thithi (lunar day) based on precise phase extrema
- Paksha (fortnight) transitions: Krishna (Purnima→Amavasya), Shukla (Amavasya→Purnima)
- Parva (major lunar events) identification using local minima/maxima
- Support for BCE dates and ancient calendar studies
- Visualization-ready outputs for integration with Stellarium





**Engineering Improvements:**

- Separation of concerns: calculation, caching, filtering, visualization
- Hybrid caching: joblib disk cache + lru_cache memory cache
- Modular, maintainable code structure
- Robust error handling and graceful BCE date support
- Clean interface and extensible design

In [1]:
# Core imports
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path
import warnings
from functools import lru_cache
import json
import re
import os

# Astronomical libraries
from astropy.time import Time
from astropy.coordinates import get_body, AltAz, EarthLocation, GeocentricTrueEcliptic
from astroplan import Observer
import astropy.units as u

# Scientific computing
from scipy.signal import argrelextrema
from joblib import Memory

# Visualization
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display, HTML
import pprint
from glob import glob

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

print("Lunar Events Calculator - Modern Astronomical Implementation")
print("=" * 60)

Lunar Events Calculator - Modern Astronomical Implementation


In [2]:
class LunarConfig:
    """Configuration constants for lunar calculations."""
    
    # Default locations
    LOCATIONS = {
        'kurukshetra': EarthLocation(lat=29.9697*u.deg, lon=76.8786*u.deg, height=0*u.m),
        'delhi': EarthLocation(lat=28.6139*u.deg, lon=77.2090*u.deg, height=216*u.m),
        'mumbai': EarthLocation(lat=19.0760*u.deg, lon=72.8777*u.deg, height=14*u.m),
        'varanasi': EarthLocation(lat=25.3176*u.deg, lon=82.9739*u.deg, height=80*u.m),
        'ujjain': EarthLocation(lat=23.1765*u.deg, lon=75.7885*u.deg, height=494*u.m)
    }
    
    # Astronomical constants
    MOON_DIAMETER_KM = 3474.8
    LUNAR_MONTH_DAYS = 29.53059
    CACHE_DIR = './cache_lunar~'
    
    # Thithi names
    THITHI_NAMES = [
        'Pratipada', 'Dwitiya', 'Tritiya', 'Chaturthi', 'Panchami',
        'Shashthi', 'Saptami', 'Ashtami', 'Navami', 'Dashami',
        'Ekadashi', 'Dwadashi', 'Trayodashi', 'Chaturdashi', 'Purnima/Amavasya'
    ]
    
    @classmethod
    def get_location(cls, name):
        """Get predefined location by name."""
        return cls.LOCATIONS.get(name.lower(), cls.LOCATIONS['kurukshetra'])
    
    @classmethod
    def thithi_name(cls, thithi_number):
        """Get traditional name for thithi number (1-15)."""
        if 1 <= thithi_number <= 15:
            return cls.THITHI_NAMES[thithi_number - 1]
        return f"Invalid_Thithi_{thithi_number}"

In [3]:
class LunarDataCalculator:
    """Core calculator for lunar astronomical data with elegant caching."""
    
    def __init__(self, location=None, cache_dir=None):
        self.location = location if location is not None else LunarConfig.get_location('kurukshetra')
        self.observer = Observer(location=self.location, timezone="Asia/Kolkata")
        
        # Setup disk cache
        cache_path = cache_dir or LunarConfig.CACHE_DIR
        os.makedirs(cache_path, exist_ok=True)
        self.memory = Memory(location=cache_path, verbose=0)
        
        # Create cached versions of methods
        self._calculate_moon_data_disk = self.memory.cache(self._calculate_moon_data_raw)
    
    def _calculate_moon_data_raw(self, start_time_jd, ndays, event_type='set'):
        """Raw calculation method - cached to disk."""
        start_time = Time(start_time_jd, format='jd')
        data = []
        
        current_time = start_time
        for day in range(ndays):
            try:
                # Calculate moon event
                if event_type == 'set':
                    moon_event = self.observer.moon_set_time(current_time + 0.5*u.day, which='next')
                else:
                    moon_event = self.observer.moon_rise_time(current_time + 0.5*u.day, which='next')
                
                # Get astronomical data
                phase = self.observer.moon_illumination(moon_event)
                moon = get_body('moon', moon_event, self.location)
                sun = get_body('sun', moon_event, self.location)
                
                # Coordinate transformations
                moon_altaz = moon.transform_to(AltAz(obstime=moon_event, location=self.location))
                moon_ecliptic = moon.transform_to(GeocentricTrueEcliptic(equinox=moon_event))
                
                # Format time for display (handle BCE dates gracefully)
                try:
                    local_time = (moon_event + 5.5*u.hour).iso  # IST approximation
                except (ValueError, OverflowError):
                    local_time = moon_event.iso  # Fallback to UTC for BCE dates
                
                data.append({
                    'JD': moon_event.jd,
                    'UTC': moon_event.iso,
                    'LocalTime': local_time,
                    'Phase': phase * 100,  # Convert to percentage
                    'Altitude': moon_altaz.alt.deg,
                    'Azimuth': moon_altaz.az.deg,
                    'Longitude': moon_ecliptic.lon.deg,
                    'Declination': moon.dec.deg,
                    'Distance_km': moon.distance.km,
                    'Elongation': moon.separation(sun).deg
                })
                
                current_time = moon_event
                
            except Exception as e:
                print(f"Warning: Error calculating day {day}: {e}")
                current_time += 1*u.day
                continue
        
        return pd.DataFrame(data)
    
    @lru_cache(maxsize=128)
    def get_moon_data(self, start_time_jd, ndays=60, event_type='set'):
        """Get moon data with hybrid caching (disk + memory)."""
        return self._calculate_moon_data_disk(start_time_jd, ndays, event_type)
    
    def clear_cache(self):
        """Clear both memory and disk caches."""
        self.get_moon_data.cache_clear()
        self.memory.clear()

In [4]:
class LunarCalendarAnalyzer:
    """Astronomically accurate lunar calendar analyzer using phase extrema."""
    
    def __init__(self, data_calculator):
        self.calculator = data_calculator
    
    def analyze_lunar_events(self, moon_data_df):
        """Analyze lunar events using correct astronomical principles."""
        if len(moon_data_df) < 10:
            raise ValueError("Need at least 10 data points for reliable phase analysis")
        
        phase_array = moon_data_df['Phase'].values
        
        # Find phase extrema (the correct way to identify lunar events)
        local_minima_idx = argrelextrema(phase_array, np.less, order=2)[0]
        local_maxima_idx = argrelextrema(phase_array, np.greater, order=2)[0]
        
        # Calculate Thithis based on position relative to extrema
        moon_data_with_thithis = self._calculate_thithis(moon_data_df, local_minima_idx, local_maxima_idx)
        
        # Get the actual events AFTER Thithi calculation (so they have all columns)
        amavasya_events = moon_data_with_thithis.iloc[local_minima_idx].copy()
        purnima_events = moon_data_with_thithis.iloc[local_maxima_idx].copy()
        
        return {
            'full_data': moon_data_with_thithis,
            'amavasya': amavasya_events,
            'purnima': purnima_events,
            'phase_extrema': {
                'minima_indices': local_minima_idx,
                'maxima_indices': local_maxima_idx
            }
        }
    
    def _calculate_thithis(self, df, minima_idx, maxima_idx):
        """Calculate Thithis based on astronomical phase extrema."""
        result_df = df.copy()
        result_df['Thithi'] = 1  # Initialize to valid Thithi (changed from 0)
        result_df['Paksha'] = ''
        result_df['Parva'] = ''
        
        # Combine and sort all extrema
        all_extrema = []
        for idx in minima_idx:
            all_extrema.append((idx, 'Amavasya'))
        for idx in maxima_idx:
            all_extrema.append((idx, 'Purnima'))
        
        all_extrema.sort(key=lambda x: x[0])
        
        if len(all_extrema) == 0:
            # No extrema found, assign default values
            result_df['Paksha'] = 'Unknown'
            result_df['Parva'] = '-'
            return result_df
        
        # Handle rows BEFORE first extrema (extend first cycle backward)
        if all_extrema[0][0] > 0:
            first_extrema_idx, first_extrema_type = all_extrema[0]
            # Determine what kind of Paksha leads to this extrema
            if first_extrema_type == 'Amavasya':
                # Before Amavasya should be Krishna Paksha ending
                paksha = 'Krishna'
            else:  # Purnima
                # Before Purnima should be Shukla Paksha ending
                paksha = 'Shukla'
            
            # Assign Thithis to rows before first extrema
            for idx in range(first_extrema_idx):
                # Calculate Thithi based on distance from extrema
                distance_from_extrema = first_extrema_idx - idx
                thithi = max(1, min(14, 15 - distance_from_extrema))
                result_df.iloc[idx, result_df.columns.get_loc('Thithi')] = thithi
                result_df.iloc[idx, result_df.columns.get_loc('Paksha')] = paksha
                result_df.iloc[idx, result_df.columns.get_loc('Parva')] = '-'
        
        # Assign Thithis and Pakshas for main cycles
        for i, (extrema_idx, extrema_type) in enumerate(all_extrema):
            # Mark the extrema itself
            result_df.iloc[extrema_idx, result_df.columns.get_loc('Thithi')] = 15
            result_df.iloc[extrema_idx, result_df.columns.get_loc('Parva')] = extrema_type
            
            # Find the range to next extrema
            if i < len(all_extrema) - 1:
                next_extrema_idx = all_extrema[i + 1][0]
                next_extrema_type = all_extrema[i + 1][1]
                
                # Determine Paksha based on transition
                if extrema_type == 'Amavasya' and next_extrema_type == 'Purnima':
                    paksha = 'Shukla'
                elif extrema_type == 'Purnima' and next_extrema_type == 'Amavasya':
                    paksha = 'Krishna'
                else:
                    paksha = 'Unknown'
                
                # Calculate Thithis in the range (CORRECTED ALGORITHM)
                range_indices = list(range(extrema_idx + 1, next_extrema_idx))
                
                # Assign Thithis 1-14 proportionally across the range
                for j, idx in enumerate(range_indices):
                    if idx < len(result_df):
                        # Map position proportionally to Thithi 1-14
                        if len(range_indices) > 0:
                            thithi = min(14, max(1, round((j + 1) * 14 / len(range_indices))))
                        else:
                            thithi = 1
                        
                        result_df.iloc[idx, result_df.columns.get_loc('Thithi')] = thithi
                        result_df.iloc[idx, result_df.columns.get_loc('Paksha')] = paksha
                        result_df.iloc[idx, result_df.columns.get_loc('Parva')] = '-'
                
                # Set Paksha for the extrema itself
                result_df.iloc[extrema_idx, result_df.columns.get_loc('Paksha')] = paksha
        
        # Handle rows AFTER last extrema (extend last cycle forward)
        if len(all_extrema) > 0 and all_extrema[-1][0] < len(result_df) - 1:
            last_extrema_idx, last_extrema_type = all_extrema[-1]
            # Determine what kind of Paksha follows this extrema
            if last_extrema_type == 'Amavasya':
                # After Amavasya should be Shukla Paksha beginning
                paksha = 'Shukla'
            else:  # Purnima
                # After Purnima should be Krishna Paksha beginning
                paksha = 'Krishna'
            
            # Assign Thithis to rows after last extrema
            for idx in range(last_extrema_idx + 1, len(result_df)):
                # Calculate Thithi based on distance from extrema
                distance_from_extrema = idx - last_extrema_idx
                thithi = min(14, max(1, distance_from_extrema))
                result_df.iloc[idx, result_df.columns.get_loc('Thithi')] = thithi
                result_df.iloc[idx, result_df.columns.get_loc('Paksha')] = paksha
                result_df.iloc[idx, result_df.columns.get_loc('Parva')] = '-'
        
        return result_df

    @classmethod
    def get_specific_events(cls, lunar_analysis, paksha, thithi):
        """Get specific lunar events (e.g., 'shukla_pratipada', 'krishna_chaturthi')."""
        full_data = lunar_analysis['full_data']
        event_filter = ((full_data['Paksha'].str.lower() == paksha.lower()) & (full_data['Thithi'] == thithi))
        return full_data[event_filter].copy()
        
        # event_filters = {
        #     'shukla_pratipada': (full_data['Paksha'] == 'Shukla') & (full_data['Thithi'] == 1),
        #     'shukla_dwitiya': (full_data['Paksha'] == 'Shukla') & (full_data['Thithi'] == 2),
        #     'krishna_pratipada': (full_data['Paksha'] == 'Krishna') & (full_data['Thithi'] == 1),
        #     'krishna_trayodashi': (full_data['Paksha'] == 'Krishna') & (full_data['Thithi'] == 13),
        #     'krishna_chaturdashi': (full_data['Paksha'] == 'Krishna') & (full_data['Thithi'] == 14),
        #     'amavasya': full_data['Parva'] == 'Amavasya',
        #     'purnima': full_data['Parva'] == 'Purnima'
        # }
        
        # if event_type not in event_filters:
        #     raise ValueError(f"Unknown event type: {event_type}. Available: {list(event_filters.keys())}")
        
        
        # return full_data[event_filters[event_type]].copy()

In [5]:
class StellariumIntegration:
    """Integration with Stellarium for visualization and film strips."""
    
    def __init__(self, base_dir='../stel_scripts', file_prefix='lunar'):
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(exist_ok=True)
        self.file_prefix = file_prefix


    def export_events_for_stellarium(self, events_dict, event_type='set', start_time=None):
        """Export lunar events as JD values for Stellarium scripts."""
        jd_dict = {}

        for event_name, event_df in events_dict.items():
            if not event_df.empty:
                jd_dict[event_name] = event_df['JD'].tolist()
        
        # Write to include file
        year = start_time.ymdhms.year if start_time else -99999
        inc_file = self.base_dir / f"{self.file_prefix}-{event_type}-events-{year:05d}.inc"
        with open(inc_file, 'w') as f:
            f.write(f"MOON_{event_type.upper()}_JDS = ")
            pprint.pprint(jd_dict, stream=f, width=120, compact=True)
        
        print(f"Exported {len(jd_dict)} event types to {inc_file}")
        return inc_file

    def create_photo_strip(self, lunar_analysis, event_type='set', start_time=None ):
        """Create photo strip from Stellarium-generated images."""
        year = start_time.ymdhms.year if start_time else -99999
        images_dir_root = self.base_dir / "moon-shranga~" / event_type / str(year)

        images_dir_root = Path(images_dir_root)
        if not images_dir_root.exists():
            print(f"Images directory not found: {images_dir_root}")
            return None
        
        event_filters = {
            'shukla_pratipada': lambda : LunarCalendarAnalyzer.get_specific_events(lunar_analysis, 'shukla', 1),
            'shukla_dwitiya': lambda : LunarCalendarAnalyzer.get_specific_events(lunar_analysis, 'shukla', 2),
            'purnima': lambda : lunar_analysis['purnima'],
            'krishna_chaturdashi': lambda : LunarCalendarAnalyzer.get_specific_events(lunar_analysis, 'krishna', 14),
            'amavasya': lambda : lunar_analysis['amavasya'],
        }

        ans_files = []
        for images_dir in images_dir_root.iterdir():

            if not images_dir.is_dir():
                # print(f"Skipping non-directory: {images_dir}")
                continue

            # Get image files
            image_files = sorted(list(images_dir.glob("*.jpg")) + list(images_dir.glob("*.png")))
            
            if not image_files:
                print(f"No images found in {images_dir}")
                continue

            event_df = event_filters[images_dir.name]()

            if event_df.empty:
                print(f"No events found for {images_dir.name}")
                continue

            # Create grid
            n_images = min(len(image_files), len(event_df))
            cols = min(7, n_images)
            rows = (n_images + cols - 1) // cols
            
            fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 2.6*rows))
            if rows == 1:
                axes = axes.reshape(1, -1)
            elif cols == 1:
                axes = axes.reshape(-1, 1)
            
            for i in range(rows * cols):
                row, col = i // cols, i % cols
                ax = axes[row, col]
                
                if i < n_images:
                    # Load and display image
                    img = Image.open(image_files[i])
                    # negate the image
                    
                    # clip the bottom 5% of the image
                    img = img.crop((0, 0, img.width, int(img.height * 0.95)))  # Crop bottom 5%

                    # img = img.astype(np.uint8)
                    ax.imshow(img)
                    
                    # Add title with event data
                    # event_data = event_df.iloc[i]
                    # title = f"JD: {event_data['JD']:.1f}\nPhase: {event_data['Phase']:.1f}%"
                    # if 'Thithi' in event_data:
                    #     thithi_name = LunarConfig.thithi_name(int(event_data['Thithi']))
                    #     title += f"\n{event_data['Paksha']} {thithi_name}"
                    
                    # ax.set_title(title, fontsize=10)
                
                ax.axis('off')
            
            plt.tight_layout()
            
            # Save strip
            output_file = images_dir_root/ f"strip-{event_type}-{images_dir.name}-{year}.jpg"
            fig.savefig(output_file, dpi=300, bbox_inches='tight')
            ans_files.append(output_file)

            plt.show()
            print(f"Photo strip saved to: {output_file}")
        
        return ans_files

In [ ]:
class LunarEventsCalculator:
    """Main interface for lunar events calculation with elegant design."""
    
    def __init__(self, location=None, cache_dir=None):
        """Initialize the lunar events calculator.
        
        Args:
            location: EarthLocation or location name string
            cache_dir: Directory for caching calculations
        """
        # Handle location parameter
        if isinstance(location, str):
            location = LunarConfig.get_location(location)
        
        # Initialize components
        self.data_calculator = LunarDataCalculator(location, cache_dir)
        self.calendar_analyzer = LunarCalendarAnalyzer(self.data_calculator)
        self.stellarium = StellariumIntegration()
        
        # Store configuration
        self.location = self.data_calculator.location
        self.observer = self.data_calculator.observer
    
    def calculate_events(self, start_time, ndays=60, event_type='set'):
        """Calculate lunar events for a given time period.
        
        Args:
            start_time: astropy.time.Time, datetime, or string
            ndays: Number of days to calculate
            event_type: 'set' or 'rise'
            
        Returns:
            Dictionary with lunar analysis results
        """
        # Convert start_time to Time object if needed
        if not isinstance(start_time, Time):
            if isinstance(start_time, str):
                # Handle BCE dates and various formats
                try:
                    if start_time.startswith('-'):
                        start_time = Time(start_time, format='fits', scale='utc')
                    else:
                        start_time = Time(start_time)
                except Exception as e:
                    raise ValueError(f"Could not parse time '{start_time}': {e}")
            else:
                start_time = Time(start_time)
        
        # Get raw moon data
        print(f"Calculating lunar events from {start_time.iso} for {ndays} days...")
        moon_data = self.data_calculator.get_moon_data(start_time.jd, ndays, event_type)
        
        if moon_data.empty:  # FLAG ERR CHECK TODO
            raise ValueError("No moon data calculated")
        
        # Analyze lunar calendar events
        lunar_analysis = self.calendar_analyzer.analyze_lunar_events(moon_data)
        
        print(f"Found {len(lunar_analysis['amavasya'])} Amavasya and {len(lunar_analysis['purnima'])} Purnima events")
        self.event_type = event_type
        self.start_time = start_time
        self.ndays = ndays
        
        return lunar_analysis
    
    def get_event_summary(self, lunar_analysis):
        """Get summary of all available lunar events."""
        event_types = [ 
            ('shukla' , 1) , ('shukla', 2), ('shukla', 15),
            ('krishna', 14), ('krishna', 15),
        ]
        
        summary = {}
        for paksha, thithi in event_types:
            try:
                events = self.calendar_analyzer.get_specific_events(lunar_analysis, paksha, thithi)
                summary[(paksha, thithi)] = len(events)
            except Exception as e:
                summary[(paksha, thithi)] = f"Error: {e}"
        
        return summary
    
    def shukla_pratipada_events(self, start_time, ndays=60):
        """Get Shukla Pratipada events (1st day after Amavasya)."""
        analysis = self.calculate_events(start_time, ndays)
        return self.calendar_analyzer.get_specific_events(analysis, 'shukla', 1)
    
    def krishna_chaturthi_events(self, start_time, ndays=60):
        """Get Krishna Chaturthi events (4th day after Purnima)."""
        analysis = self.calculate_events(start_time, ndays)
        return self.calendar_analyzer.get_specific_events(analysis, 'krishna', 4)
    
    def krishna_chaturdashi_events(self, start_time, ndays=60):
        """Get Krishna Chaturdashi events (14th day after Purnima)."""
        analysis = self.calculate_events(start_time, ndays)
        return self.calendar_analyzer.get_specific_events(analysis, 'krishna', 14)
    
    def amavasya_events(self, start_time, ndays=60):
        """Get Amavasya events (New Moon)."""
        analysis = self.calculate_events(start_time, ndays)
        return self.calendar_analyzer.get_specific_events(analysis, 'krishna', 15)  # Amavasya is always Thithi 15 in Krishna Paksha
    
    def purnima_events(self, start_time, ndays=60):
        """Get Purnima events (Full Moon)."""
        analysis = self.calculate_events(start_time, ndays)
        return self.calendar_analyzer.get_specific_events(analysis, 'shukla', 15)  # Purnima is always Thithi 15 in Shukla Paksha

    def export_for_stellarium(self, lunar_analysis):
        """Export events for Stellarium visualization."""
        events_dict = {
            'amavasya': lunar_analysis['amavasya'],
            'purnima': lunar_analysis['purnima']
        }
        
        # Add specific event types
        for thithi_tag , paksha, thithi in [
            ('shukla_pratipada', 'shukla', 1), 
            ('shukla_dwitiya', 'shukla', 2), 
            ('krishna_chaturdashi', 'krishna', 14)
        ]:
            try:
                events_dict[thithi_tag] = self.calendar_analyzer.get_specific_events(lunar_analysis, paksha, thithi)
            except Exception:
                pass

        export_file = self.stellarium.export_events_for_stellarium(events_dict,  event_type=self.event_type, start_time=self.start_time)
        print(f"\nExported events for Stellarium to: {export_file}")
       # show the exported file
        with open(export_file, 'r') as f:
            print(f.read()[:500])  # Show first 500 characters for brevity

        export_file_name_part = str(export_file).split('/')[-1]
        year = self.start_time.ymdhms.year if self.start_time else -99999
        event_type = self.event_type if self.event_type else 'set'

        # Create directory for Stellarium images
        images_dir = self.stellarium.base_dir / f"moon-shranga~/{event_type}/{year}"
        images_dir.mkdir(parents=True, exist_ok=True)
        # Create thithi_tag directories under images_dir
        for thithi_tag in events_dict.keys():
            thithi_dir = images_dir / thithi_tag
            thithi_dir.mkdir(parents=True, exist_ok=True)

        # Ask the user to run the Stellarium script manually , so the pictures can be generated
        display(HTML(f"""
                    <big>
                    <hr>
                    <li>Ensure <b>../stel_scripts/a9-moon-shranga.ssc</b> includes this <code>include ({export_file_name_part});</code></li> <br>
                    <li>Run the Stellarium script <b>../stel_scripts/a9-moon-shranga.ssc</b> to manually to generate the pictures.</li> <br>
                    <li>The script will generate images in the <code>../stel_scripts/moon-shranga~/{event_type}/{year}</code> directory.</li> <br>
                    <li>After running the script, run the next cell to create the photo strips.</li>
                    <hr>
                    </big>
        """))

        return export_file
    
    def create_visualization(self, lunar_analysis):
        #../stel_scripts/moon-shranga~/set/-0500
        """Create photo strip visualization if images are available."""
        return self.stellarium.create_photo_strip(lunar_analysis, self.event_type, self.start_time)

    def clear_cache(self):
        """Clear all caches."""
        self.data_calculator.clear_cache()
        print("Cache cleared")

    def generate_strips_for_all_leaf_dirs(self, base_dir=None, show_progress=True, force=False, clip=None, dirmask=r"."):
        if clip == True : clip = 2
        """Generate photo strips for all leaf directories under moon-shranga~.
        
        This method finds all leaf directories (directories containing images but no 
        subdirectories with images) under the moon-shranga~ directory structure and 
        creates photo strips for each one without rendering them in the notebook.
        
        Args:
            base_dir: Base directory to search (defaults to stellarium base_dir/moon-shranga~)
            show_progress: Whether to show progress information
            
        Returns:
            List of generated strip file paths
        """
        if base_dir is None:
            base_dir = self.stellarium.base_dir / "moon-shranga~"
        else:
            base_dir = Path(base_dir)
        
        if not base_dir.exists():
            print(f"Base directory not found: {base_dir}")
            return []
        
        # Find all leaf directories (directories with image files but no subdirectories with images)
        leaf_dirs = []
        for root, dirs, files in os.walk(base_dir):
            root_path = Path(root)
            
            # Check if this directory has image files
            has_images = any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files)
            
            # Check if any subdirectories have images
            has_subdir_with_images = False
            for subdir in dirs:
                subdir_path = root_path / subdir
                if any(subdir_path.glob('*.jpg')) or any(subdir_path.glob('*.png')):
                    has_subdir_with_images = True
                    break
            
            # If this directory has images but no subdirectories with images, it's a leaf
            if has_images and not has_subdir_with_images:
                leaf_dirs.append(root_path)
        
        if show_progress:
            print(f"Found {len(leaf_dirs)} leaf directories with images")
            for leaf_dir in leaf_dirs:
                print(f"  - {leaf_dir}")
        
        generated_strips = []
        
        for leaf_dir in leaf_dirs:
            # hack : process only set of -500

            # if not "/-500/" in str(leaf_dir): continue
            # if not "pratipada" in str(leaf_dir): continue
            if re.match(dirmask, str(leaf_dir)) is None:
                continue

            try:
                # Get image files
                image_files = sorted(list(leaf_dir.glob("??.jpg")) + list(leaf_dir.glob("??.png")))
                image_files = [f for f in image_files if f.stat().st_size > 0]  # Filter out empty files
                image_files = [f for f in image_files if re.match(r'.*\d\d.jpg.*', str(f))]  # keep only files with two digit numbers in name
                
                if not image_files:
                    if show_progress:
                        print(f"No images found in {leaf_dir}")
                    continue

                # print(f"Processing directory: {leaf_dir} with {len(image_files)} images\n {image_files[:3]} ... {image_files[-3:]}")

                # Generate output filename
                relative_path = leaf_dir.relative_to(base_dir)
                safe_name = str(relative_path).replace('/', '-').replace('\\', '-')
                prefix = f"clip{clip:02d}-strip-" if clip else "strip-"
                output_file = leaf_dir.parent / f"{prefix}{safe_name}.jpg"

                # skip if output file already exists and is not empty 
                if not force and output_file.exists() and output_file.stat().st_size > 0:
                    if show_progress:
                        print(f"Output file already exists and is not empty: {output_file}")
                    generated_strips.append(output_file)
                    continue

                n_cols = 7
                # limit images to fit n_cols
                image_files = image_files[:n_cols * (len(image_files) // n_cols)]

                # Create grid layout
                n_images = len(image_files)
                cols = min(n_cols, n_images)
                rows = (n_images + cols - 1) // cols
                
                if show_progress:
                    print(f"Processing {leaf_dir.name}: {n_images} images in {rows}x{cols} grid")
                
                # Create figure without displaying it
                fig, axes = plt.subplots(rows, cols, figsize=(2.9*cols/(1.8 if clip else 1), 2.3*rows/(1.2 if clip else 1)))
                
                # Handle different subplot configurations
                if rows == 1 and cols == 1:
                    axes = [axes]  # Make it iterable
                elif rows == 1:
                    axes = axes.reshape(1, -1)
                elif cols == 1:
                    axes = axes.reshape(-1, 1)
                
                for i in range(rows * cols):
                    # Calculate position
                    if rows == 1 and cols == 1:
                        ax = axes[0]
                    elif rows == 1:
                        ax = axes[0, i]
                    elif cols == 1:
                        ax = axes[i, 0]
                    else:
                        row, col = i // cols, i % cols
                        ax = axes[row, col]
                    
                    if i < n_images:
                        # Load and display image
                        img = Image.open(image_files[i])
                        # Crop the bottom 5% of the image
                        img = img.crop((0, 0, img.width, int(img.height * 0.95)))
                        # negate the image
                        # img = ImageOps.invert(img.convert('RGB'))
                        original_size = img.size    
                        
                        # clip grabs 1/N the image from the center on all sides
                        if clip:
                            w = img.width 
                            h = img.height
                            W = w//clip  # Width after clipping
                            H = h//clip  # Height after clipping
                            left = ((w - W) // 2)*1.13
                            top = ((h - H) // 2)*1.2
                            right = left + W*.6
                            bottom = top + H*1.2
                            # bottom = .95 * h       # 95% of the height (bottom edge)
                            img = img.crop((left, top, right, bottom))

                            # negate the image
                            # img = ImageOps.invert(img.convert('RGB'))
                            # img = img.convert('L')  # convert to grayscale
                            # img = ImageOps.autocontrast(img, cutoff=2)  # auto contrast with 2% cutoff
                            # img = ImageOps.equalize(img)  # histogram equalization

                        crop_size = img.size
                        if ( i == 0) :
                            print(f" {i} : clip:{clip} : ✓ Cropped size: {original_size} => {crop_size} (width x height) ")

                        #  I need to add  Column header A, B, C ... G on top of each column
                        if i < n_cols:
                            ax.annotate(chr(65 + i), xy=(0.5, 1.05), xycoords='axes fraction', ha='center', va='bottom', fontsize=10, fontweight='normal', rotation=0)

                        # I need to add Row header 1, 2, 3 ... on left side of each row
                        if i % n_cols == 0:
                            ax.annotate(str(1 + (i // n_cols)), xy=(-0.1, 0.5), xycoords='axes fraction', ha='right', va='center', fontsize=10, fontweight='normal', rotation=0)

                        ax.imshow(img)
                        # save the img
                        clip_file = leaf_dir / f"{leaf_dir.name}_{i:02}_clip{clip if clip else 1}.jpg"
                        print(f" {i} : clip:{clip} : ✓ Saved clipped image: {clip_file}")          
                        img.save(clip_file, "JPEG")

                    ax.axis('off')
                
                plt.tight_layout()
                
                # Generate output filename
                # relative_path = leaf_dir.relative_to(base_dir)
                # safe_name = str(relative_path).replace('/', '-').replace('\\', '-')
                # output_file = leaf_dir.parent / f"strip-{safe_name}.jpg"
                
                # Save strip without displaying
                fig.savefig(output_file, dpi=300, bbox_inches='tight')
                plt.close(fig)  # Close figure to free memory
                
                generated_strips.append(output_file)
                
                if show_progress:
                    print(f"  ✓ Generated strip: {output_file}")
                    
            except Exception as e:
                if show_progress:
                    print(f"  ✗ Error processing {leaf_dir}: {e}")
                continue
        
        if show_progress:
            print(f"\n🎉 Generated {len(generated_strips)} photo strips total")
        
        return generated_strips

In [19]:
calculator = LunarEventsCalculator(location='kurukshetra')
# bce_500_set_3years = calculator.calculate_events('-00500-01-01T00:00:00.000', ndays=366*3, event_type='set')
# Export events for Stellarium
# export_file = calculator.export_for_stellarium(bce_500_set_3years)

# bce_509_set_3years = calculator.calculate_events('-00509-01-01T00:00:00.000', ndays=366*3, event_type='set')
# Export events for Stellarium
# export_file = calculator.export_for_stellarium(bce_509_set_3years)

# bce_set_4years = calculator.calculate_events('2025-01-01T00:00:00.000', ndays=366*4, event_type='set')
# Export events for Stellarium
# export_file = calculator.export_for_stellarium(bce_set_4years)

# pause - wait for user input
# input("Press Enter after running the Stellarium script to generate images...")

# generate photo strips
# strips = calculator.create_visualization(bce_500_set_3years)
# strips

In [20]:
from responses import quote


for clipx in [4,False][:1] :
    # generated_strips = calculator.generate_strips_for_all_leaf_dirs(show_progress=not True, clip=clipx, force=not False, dirmask=r".*moon.8.alt.*(shukla|purnima).*")
    generated_strips = calculator.generate_strips_for_all_leaf_dirs(show_progress=not True, clip=clipx, force=not False, dirmask=r".*set/.*2025/.*(shukla|purnima).*")
    print(f"Generated {len(generated_strips)} strips with clip={clipx}\n {generated_strips}")

    # Minimal inline SVG (Wikipedia-style external link icon)
    external_svg = (
        '<svg xmlns="http://www.w3.org/2000/svg" '
        'width="12" height="12" viewBox="0 0 24 24" '
        'fill="none" stroke="currentColor" stroke-width="2" '
        'stroke-linecap="round" stroke-linejoin="round" '
        'style="vertical-align: text-bottom;">'
        '<path d="M18 13v6a2 2 0 0 1-2 2H6'
        'a2 2 0 0 1-2-2V8a2 2 0 0 1 2-2h6" />'
        '<polyline points="15 3 21 3 21 9" />'
        '<line x1="10" y1="14" x2="21" y2="3" />'
        '</svg>'
    )
    # render each strip as an anchor
    # for strip in generated_strips: display(HTML(f"<a href='{strip}'>{strip}</a>"))

    Thithi_ranks = {
        'amavasya': 0, 'shukla_pratipada': 1, 'shukla_dwitiya': 2,
        'purnima': 15, 'krishna_chaturdashi': 15+14,
    }

    import random
    pvt = pd.DataFrame([str(f).split('/')[-3:-1] + [f.name.split('-')[-1][:-4]] + [f.name] + [str(f)] for f in generated_strips], 
                columns=['Event',  'Tag', 'Phase', 'File', 'Path']
    ).assign(
        Clip=lambda x:  x['File'].str.contains('clip').apply(lambda y: 'clip' if y else 'full'),
        PathAnchor = lambda x: x.apply(lambda row: f"<big><a href='{row['Path']}'>{external_svg}</a></big>", axis=1),
        PathImg=lambda x: x.apply(
            lambda row: 
                f'{row["PathAnchor"]}'
                f'<a href="{row["Path"]}">'
                f'<br>{row["Path"]}</br>'
                f'<img src="./{row["Path"]}?cb={random.randint(0, 10000)}"/>'
                # f'width="600" height="100"'
                # f'style="object-fit:cover;"/>'
                f'</a>',
            axis=1
        ),
        Thithi = lambda x: x['Phase'].map(lambda e: Thithi_ranks.get(e, None)),
    )[
        lambda x: x.Thithi.isin([0*111+1, 0*111+2, 0*111+15,])
    ].pivot_table(
    index=['Event', 'Phase', 'Thithi'], columns=['Tag', 'Clip'], values='PathImg', aggfunc=lambda x: x
    ).sort_values(by=['Event','Thithi'], ascending=[False, True])

    # I need to display the pivot table with PathAnchors as HTML links
    html = HTML(pvt.to_html(escape=False))
    # save to file and display
    with open("lunar_events_pivot~.html", "w") as f:
        f.write(html.data)
    display(html)



 0 : clip:4 : ✓ Cropped size: (3600, 2139) => (541, 641) (width x height) 
 0 : clip:4 : ✓ Saved clipped image: ../stel_scripts/moon-shranga~/set/2025/shukla_dwitiya/shukla_dwitiya_00_clip4.jpg
 1 : clip:4 : ✓ Saved clipped image: ../stel_scripts/moon-shranga~/set/2025/shukla_dwitiya/shukla_dwitiya_01_clip4.jpg
 2 : clip:4 : ✓ Saved clipped image: ../stel_scripts/moon-shranga~/set/2025/shukla_dwitiya/shukla_dwitiya_02_clip4.jpg
 3 : clip:4 : ✓ Saved clipped image: ../stel_scripts/moon-shranga~/set/2025/shukla_dwitiya/shukla_dwitiya_03_clip4.jpg
 4 : clip:4 : ✓ Saved clipped image: ../stel_scripts/moon-shranga~/set/2025/shukla_dwitiya/shukla_dwitiya_04_clip4.jpg
 5 : clip:4 : ✓ Saved clipped image: ../stel_scripts/moon-shranga~/set/2025/shukla_dwitiya/shukla_dwitiya_05_clip4.jpg
 6 : clip:4 : ✓ Saved clipped image: ../stel_scripts/moon-shranga~/set/2025/shukla_dwitiya/shukla_dwitiya_06_clip4.jpg
 7 : clip:4 : ✓ Saved clipped image: ../stel_scripts/moon-shranga~/set/2025/shukla_dwitiya/s

In [ ]:
# Now let's create a photo strip for all Purnima events in the last 3 years of BCE 500
strips = calculator.create_visualization(bce_500_set_3years)

## Example Usage

Now let's demonstrate the calculator with both modern and historical dates:

In [ ]:
# Create calculator for Kurukshetra
calculator = LunarEventsCalculator(location='kurukshetra')

print("Lunar Events Calculator initialized for Kurukshetra")
print(f"Location: {calculator.location}")
print(f"Observer timezone: {calculator.observer.timezone}")

In [ ]:
# Sample Use: Modern date calculation
print("=== Modern Date Example ===")
modern_analysis = calculator.calculate_events('2025-08-01', ndays=90)

# Get summary
summary = calculator.get_event_summary(modern_analysis)
print("\nEvent Summary:")
for event, count in summary.items():
    print(f"  {event}: {count}")

# Display some events
print("\nAmavasya Events:")
display(modern_analysis['amavasya'][['JD', 'UTC', 'LocalTime', 'Phase', 'Thithi', 'Paksha', 'Parva']].head().style.format(precision=2).set_caption("Amavasya Events"))

print("\nPurnima Events:")
purnima_events = calculator.calendar_analyzer.get_specific_events(modern_analysis, 'purnima', 15)
display(modern_analysis['purnima'][['JD', 'UTC', 'LocalTime', 'Phase', 'Thithi', 'Paksha', 'Parva']].head().style.format(precision=2).set_caption("Purnima Events"))


print("\nShukla Pratipada Events:")
shukla_pratipada = calculator.calendar_analyzer.get_specific_events(modern_analysis, 'shukla',1)
display(shukla_pratipada[['JD', 'UTC', 'LocalTime', 'Phase', 'Thithi', 'Paksha', 'Parva']].head().style.format(precision=2).set_caption("Shukla Pratipada Events"  ))

In [ ]:
# Historical date (BCE) calculation check
print("=== Historical Date Example (500 BCE) ===")
try:
    historical_analysis = calculator.calculate_events('-00500-01-01T00:00:00.000', ndays=36*5, event_type='set')
    
    # Get summary
    historical_summary = calculator.get_event_summary(historical_analysis)
    print("\nHistorical Event Summary:")
    for event, count in historical_summary.items():
        print(f"  {event}: {count}")
    
    # Display Purnima events from 500 BCE
    display(historical_analysis['purnima'][['JD', 'UTC', 'Phase', 'Thithi', 'Paksha', 'Parva']].head(10).style.format({
        'JD': '{:.2f}', 'Phase': '{:.1f}%', 'Thithi': lambda x: LunarConfig.thithi_name(int(x)),
        'Paksha': lambda x: x.capitalize(), 'Parva': lambda x: x.capitalize()
    }).set_caption("Purnima Events from 500 BCE"))

    #Amavasya events
    display(historical_analysis['amavasya'][['JD', 'UTC', 'Phase', 'Thithi', 'Paksha', 'Parva']].head(10).style.format({
        'JD': '{:.2f}', 'Phase': '{:.1f}%', 'Thithi': lambda x: LunarConfig.thithi_name(int(x)),
        'Paksha': lambda x: x.capitalize(), 'Parva': lambda x: x.capitalize()
    }).set_caption("Amavasya Events from 500 BCE"))

    # Shukla Pratipada events
    shukla_pratipada_historical = calculator.calendar_analyzer.get_specific_events(historical_analysis, 'shukla', 1)
    if not shukla_pratipada_historical.empty:
        display(shukla_pratipada_historical[['JD', 'UTC', 'Phase', 'Thithi', 'Paksha', 'Parva']].head(10).style.format({
            'JD': '{:.2f}', 'Phase': '{:.1f}%', 'Thithi': lambda x: LunarConfig.thithi_name(int(x)),
            'Paksha': lambda x: x.capitalize(), 'Parva': lambda x: x.capitalize()
        }).set_caption("Shukla Pratipada Events from 500 BCE"))

    # Krishna Chaturdashi events
    krishna_chaturdashi = calculator.calendar_analyzer.get_specific_events(historical_analysis, 'krishna', 14)
    if not krishna_chaturdashi.empty:
        display(krishna_chaturdashi[['JD', 'UTC', 'Phase', 'Thithi', 'Paksha', 'Parva']].head(10).style.format({
            'JD': '{:.2f}', 'Phase': '{:.1f}%', 'Thithi': lambda x: LunarConfig.thithi_name(int(x)),
            'Paksha': lambda x: x.capitalize(), 'Parva': lambda x: x.capitalize()
        }).set_caption("Krishna Chaturdashi Events from 500 BCE"))

except Exception as e:
    print(f"Error with historical calculation: {e}")

In [ ]:
# Test the full data length
def sanity_check():
    # calculator.clear_cache()
    test_events = calculator.calculate_events('-00500-01-01T00:00:00.000', ndays=367*3, event_type='set')
    print(f"\nBCE 500 - 3 years data:")
    print(f"Total data points: {len(test_events['full_data'])}")
    # Check Thithi distribution
    print(f"\nFixed Thithi distribution (should be 1-15 only, NO 0):")
    thithi_counts = test_events['full_data'].Thithi.value_counts().sort_index()
    print(thithi_counts)

    # Verify no Thithi 0
    if 0 in thithi_counts.index:
        print("❌ ERROR: Still have Thithi 0!")
    else:
        print("✅ SUCCESS: No more Thithi 0!")

    # Verify exactly 15 distinct Thithis (1-15)
    distinct_thithis = set(test_events['full_data']['Thithi'].unique())
    expected_thithis = set(range(1, 16))  # {1, 2, 3, ..., 15}

    print(f"\nThithi validation:")
    print(f"  Expected: {sorted(expected_thithis)}")
    print(f"  Actual: {sorted(distinct_thithis)}")
    print(f"  Count: {len(distinct_thithis)} (should be 15)")

    if distinct_thithis == expected_thithis:
        print("✅ SUCCESS: Exactly 15 distinct Thithis (1-15)!")
    else:
        missing = expected_thithis - distinct_thithis
        extra = distinct_thithis - expected_thithis
        if missing:
            print(f"❌ Missing Thithis: {sorted(missing)}")
        if extra:
            print(f"❌ Extra Thithis: {sorted(extra)}")

    # Check distribution evenness
    thithi_1_to_14 = thithi_counts[thithi_counts.index.isin(range(1, 15))]
    if len(thithi_1_to_14) > 0:
        mean_count = thithi_1_to_14.mean()
        std_count = thithi_1_to_14.std()
        evenness = 100 * (1 - std_count/mean_count)
        print(f"\nDistribution evenness: {evenness:.1f}% (higher is better)")
        print(f"  Mean count per Thithi: {mean_count:.1f}")
        print(f"  Standard deviation: {std_count:.1f}")

    # Show edge cases are now handled
    print(f"\nEdge case verification:")
    print(f"  First 5 rows Thithis: {test_events['full_data']['Thithi'].head().tolist()}")
    print(f"  Last 5 rows Thithis: {test_events['full_data']['Thithi'].tail().tolist()}")
    print(f"  None should be 0!")

sanity_check()

In [ ]:
# ## Summary

# This new **Lunar Events Calculator** provides:

# ### ✅ **Fixed Astronomical Accuracy**
# - **Correct Thithi calculation** based on actual phase extrema (local minima/maxima)
# - **Proper Paksha logic**: Krishna = Purnima→Amavasya, Shukla = Amavasya→Purnima  
# - **Accurate Parva identification**: Purnima = phase maxima, Amavasya = phase minima
# - **True lunar day counting** from one parva to the next

# ### ✅ **Elegant Design**
# - **Separation of concerns**: Calculator, Analyzer, Stellarium integration
# - **Hybrid caching**: joblib disk cache + lru_cache memory cache
# - **Clean interface**: Simple methods for common operations
# - **Modular components**: Easy to test and maintain

# ### ✅ **Historical Date Support** 
# - **BCE date handling**: Supports negative years for ancient astronomy
# - **Graceful fallbacks**: UTC times when local time conversion fails
# - **Robust error handling**: Continues calculation despite individual failures

# ### ✅ **Preserved Functionality**
# - **Stellarium integration**: Export JD values and create photo strips
# - **Visualization methods**: Film strip generation from images
# - **Multiple locations**: Predefined astronomical sites
# - **Comprehensive data**: All astronomical coordinates and measurements

# ### 🎯 **Key Improvements Over Original**
# 1. **Astronomically correct** lunar calendar logic
# 2. **Cleaner, more maintainable** code structure  
# 3. **Better caching strategy** for performance
# 4. **Robust historical date support**
# 5. **Easier to use and extend**

# The calculator is now ready for serious astronomical research including ancient period calculations!

In [ ]:
!rm ./lunar-events-calculator.html 2>/dev/null
!date
!`which jupyter` nbconvert lunar-events-calculator.ipynb --to html --no-input --output  lunar-events-calculator.html 2>&1 | tee ~/tmp/nbconvert.log
!open ./lunar-events-calculator.html